# Первый поиск похожих случаев: TF-IDF

В этом ноутбуке одна строка датасета считается одним клиническим случаем. По тексту `description` найдём другие случаи с похожими словами и короткими фразами.

Для первого baseline используем слова и биграммы без лемматизации и удаления стоп-слов. TF-IDF строит векторы описаний, а косинусное сходство ранжирует кандидатов. Это лексический поиск: он учитывает совпадение текстовых признаков, но не понимает синонимы как языковая модель.

`bethesda` оставляем только для последующей оценки выдачи. Поиск не предсказывает категорию Bethesda. Ячейка загрузки находит корень проекта по `requirements.txt`, поэтому путь к данным не зависит от того, открыто ядро из корня проекта, папки ноутбука или `.venv`.

## 1. Загрузка корпуса

Сначала поднимаемся от текущей рабочей папки до корня проекта, распознавая его по `requirements.txt`. Затем читаем идентификатор случая, описание и категорию Bethesda. В признаки поиска ниже попадёт только `description`; категория нужна, чтобы после ранжирования сравнить результаты с категорией запроса.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "requirements.txt").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError(
            "Не удалось найти корень проекта с requirements.txt."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_first.csv"
cases = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=["case_id", "description", "bethesda", "conclusion"],
)

## 2. Проверка корпуса

Проверяем только размер таблицы и число пустых описаний. Сами тексты здесь не выводим.

In [ ]:
print(f"Строк в таблице: {len(cases)}")
print(f"Пустых description: {cases['description'].isna().sum()}")

## 3. Подготовка текстов

Убираем пропуски и описания, состоящие только из пробелов. Остальные тексты пока не лемматизируем и не фильтруем по стоп-словам: это сохраняет простой baseline.

In [ ]:
has_description = cases["description"].notna()
has_description &= cases["description"].str.strip().ne("")
search_cases = cases.loc[has_description].copy().reset_index(drop=True)

if search_cases.empty:
    raise ValueError("В корпусе нет непустых описаний для поиска.")

print(f"Описаний для поиска: {len(search_cases)}")

## 4. Построение TF-IDF-векторов

`TfidfVectorizer` создаёт признаки из отдельных слов (униграмм) и пар соседних слов (биграмм), а затем взвешивает их по TF-IDF. `fit_transform` одновременно строит словарь по корпусу и превращает все описания в разреженную матрицу: строки — случаи, столбцы — текстовые признаки.

По умолчанию векторизатор приводит текст к нижнему регистру и разбивает его на токены. Односимвольные токены он не учитывает; это ограничение базовой настройки, которое можно отдельно проверить позже.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(search_cases["description"])

print(f"Размер матрицы: {tfidf_matrix.shape[0]} случаев × {tfidf_matrix.shape[1]} признаков")

## 5. Выбор описания-запроса

Для первого примера берём описание из самого корпуса. `query_position` — его позиция в `search_cases`, начиная с нуля. Поменяй число, чтобы выбрать другой случай. Позже вместо этой строки можно будет подставить новый текст.

In [ ]:
query_position = 5
query_case = search_cases.iloc[query_position]
query_text = query_case["description"]
query_vector = vectorizer.transform([query_text])

print(f"Ищем случаи, похожие на case_id={query_case['case_id']}")
print(query_text)

## 6. Ранжирование похожих случаев

Сравниваем вектор запроса со всеми строками корпуса косинусным сходством. Чем выше оценка, тем больше совпадение TF-IDF-признаков. Строку запроса исключаем, чтобы случай не оказался самым похожим на самого себя.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(query_vector, tfidf_matrix).ravel()
similarities[query_position] = -1.0

TOP_K = 10
ranked_positions = similarities.argsort()[::-1]
ranked_positions = [
    position
    for position in ranked_positions
    if position != query_position
][:TOP_K]

results = search_cases.iloc[ranked_positions][
    ["case_id", "description", "bethesda"]
].copy()
results.insert(1, "similarity", similarities[ranked_positions])
results = results.reset_index(drop=True)

display(results)

## 7. Проверка по категории Bethesda

Пока считаем совпадение категории Bethesda с категорией запроса приближённым сигналом релевантности. Это не эталон клинической похожести и не часть поисковой модели: сравниваем категории только после ранжирования.

In [ ]:
query_bethesda = query_case["bethesda"]

if pd.isna(query_bethesda):
    print("У запроса не указана категория Bethesda; оценить совпадение категорий нельзя.")
else:
    same_bethesda = results["bethesda"].eq(query_bethesda)
    results_with_evaluation = results.assign(
        same_bethesda_as_query=same_bethesda
    )
    display(
        results_with_evaluation[
            ["case_id", "similarity", "bethesda", "same_bethesda_as_query"]
        ]
    )
    print(
        "Доля top-"
        f"{len(results)} случаев с той же Bethesda: "
        f"{same_bethesda.mean():.2f}"
    )

## Как интерпретировать результат

- `similarity` — сходство текстовых TF-IDF-векторов, а не вероятность того, что случаи клинически эквивалентны.
- `same_bethesda_as_query` показывает только совпадение категории с запросом. Совпавшая категория не гарантирует похожесть описаний.
- Просмотри тексты нескольких верхних результатов и отметь, согласуются ли они с твоей интуицией. При оценке помни: словесное совпадение может пропустить синонимы и перефразировки.
- Для следующего сравнения можно изменить один фактор, например добавить лемматизацию, и повторить поиск с тем же запросом.

## 8. Поиск через функцию `find_similar_cases`

Функция принимает текст запроса и число результатов. Она использует уже обученный на корпусе `vectorizer`, сравнивает запрос с TF-IDF-векторами и возвращает таблицу с мерой сходства, описанием, категорией Bethesda и заключением. Идентификаторы случаев в результат не включаются.

Для запроса из корпуса передай его позицию через `exclude_position`: строка исключается из кандидатов, а TF-IDF-словарь и IDF заново строятся только по оставшимся кандидатам. Это не даёт запросу участвовать в собственном поисковом индексе.

Если запрос скопирован из корпуса, функция без `exclude_position` может вернуть исходную строку: по одному тексту невозможно надёжно определить, из какой записи он взят.

In [ ]:
def find_similar_cases(text, top_k=5, exclude_position=None):
    """Найти top_k описаний, наиболее похожих на текст запроса."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text должен быть непустой строкой.")

    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k должен быть целым числом больше нуля.")

    if exclude_position is not None:
        if (
            isinstance(exclude_position, bool)
            or not isinstance(exclude_position, int)
            or not 0 <= exclude_position < len(search_cases)
        ):
            raise ValueError(
                "exclude_position должен быть допустимой позицией строки корпуса."
            )

        candidate_cases = search_cases.drop(index=exclude_position).reset_index(drop=True)
        if candidate_cases.empty:
            raise ValueError("После исключения запроса не осталось случаев для поиска.")

        query_vectorizer = TfidfVectorizer(**vectorizer.get_params())
        candidate_matrix = query_vectorizer.fit_transform(candidate_cases["description"])
    else:
        candidate_cases = search_cases
        query_vectorizer = vectorizer
        candidate_matrix = tfidf_matrix

    query_vector = query_vectorizer.transform([text.strip()])
    if query_vector.nnz == 0:
        raise ValueError("В запросе нет слов, известных TF-IDF-словарю.")

    similarities = cosine_similarity(query_vector, candidate_matrix).ravel()
    ranked_positions = similarities.argsort()[::-1][:top_k]

    results = candidate_cases.iloc[ranked_positions][
        ["description", "bethesda", "conclusion"]
    ].copy()
    results.insert(0, "similarity", similarities[ranked_positions])

    return results.reset_index(drop=True)


## 9. Пилотный поиск top-20

Берём запрос из строки корпуса с позицией `5`. Функция исключает эту строку из кандидатов. Таблица показывает только сходство, описание, категорию Bethesda и заключение.

In [ ]:
query_position = 0
query_case = search_cases.iloc[query_position]
query_text = query_case["description"]
query_bethesda = query_case["bethesda"]

if pd.isna(query_bethesda):
    raise ValueError("У выбранного запроса не указана категория Bethesda.")

top_20_results = find_similar_cases(
    text=query_text,
    top_k=20,
    exclude_position=query_position,
)

print(query_text)
display(top_20_results)

## 10. Precision@20 по Bethesda

Здесь `accuracy@20` трактуем как долю top-20 с той же категорией Bethesda. Для retrieval это точнее называть `Precision@20`; Bethesda остаётся только приближённой меткой релевантности. Сравним результат с ожидаемой долей этой категории среди всех кандидатов при случайном ранжировании.

In [ ]:
candidate_cases_for_query = search_cases.drop(index=query_position)
if candidate_cases_for_query.empty or top_20_results.empty:
    raise ValueError("Недостаточно кандидатов, чтобы вычислить Precision@20.")

precision_at_20 = float(top_20_results["bethesda"].eq(query_bethesda).mean())
random_expected_precision_at_20 = float(
    candidate_cases_for_query["bethesda"].eq(query_bethesda).mean()
)
lift_over_random = precision_at_20 - random_expected_precision_at_20

print(f"Precision@20 (совпадение Bethesda): {precision_at_20:.3f}")
print(f"Ожидание при случайном ранжировании: {random_expected_precision_at_20:.3f}")
print(f"Разница с ожиданием: {lift_over_random:+.3f}")

## 11. Запись пилотного запуска в MLflow

В запись входят имя участника, задача, дата, версия датасета (SHA-256), название модели, её параметры, размеры индекса и число запросов, метрики, файл TF-IDF-индекса и примеры top-20. Для поиска `train_size` означает количество случаев в индексе, а `test_size` — число отложенных запросов; это не обычное обучение классификатора.

Модель сохраняется как `tfidf_index.joblib`, примеры — как `top20_examples.csv`. В таблицу примеров не включаются лабораторные номера и другие идентификаторы. Для этого пилота разрешено сохранять эти артефакты из `data/processed/dataset_first.csv` только в локальный MLflow-сервер. Один запрос даёт пилотную проверку, а не устойчивую оценку качества baseline.

Клиент подключается к `http://127.0.0.1:5000` и выбирает эксперимент с ID `2` (`semantic-case-search`). Серверное хранилище логов находится в `D:\MLflow\thyroid-cytology-assistant`.

In [ ]:
from datetime import datetime, timezone
import hashlib
import tempfile
from pathlib import Path

import joblib
import mlflow
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# Set the participant name before starting the run.
PARTICIPANT_NAME = "Vildan"
if PARTICIPANT_NAME == "<SET_PARTICIPANT_NAME>":
    raise ValueError("Set PARTICIPANT_NAME before starting the MLflow run.")

DATASET_VERSION = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "search" / "TF-IDF.ipynb"
NOTEBOOK_SHA256 = hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest()
RUN_DATE_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

TRAIN_SIZE = len(candidate_cases_for_query)  # retrieval index size
TEST_SIZE = 1  # held-out query count
MODEL_NAME = "tfidf_retrieval"

synthetic_data_dir = (PROJECT_ROOT / "data" / "synthetic").resolve()
processed_dataset = (
    PROJECT_ROOT / "data" / "processed" / "dataset_first.csv"
).resolve()
dataset_path = DATA_PATH.resolve()
is_allowed_artifact_source = (
    dataset_path.is_relative_to(synthetic_data_dir)
    or dataset_path == processed_dataset
)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experiment = mlflow.set_experiment(experiment_id="2")
if experiment.name != "semantic-case-search":
    raise ValueError(
        "MLflow experiment ID 2 is not named semantic-case-search."
    )

with mlflow.start_run(run_name="tfidf-top20-single-query-pilot"):
    mlflow.log_params(
        {
            "participant_name": PARTICIPANT_NAME,
            "task_name": "similar_case_retrieval",
            "run_date_utc": RUN_DATE_UTC,
            "dataset_file": DATA_PATH.name,
            "dataset_version_sha256": DATASET_VERSION,
            "model_name": MODEL_NAME,
            "top_k": 20,
            "train_size": TRAIN_SIZE,
            "test_size": TEST_SIZE,
            "size_semantics": "train_size=index cases; test_size=held-out queries",
            "query_position": query_position,
            "self_match_excluded": True,
            "relevance_proxy": "same_bethesda",
            "baseline": "random ranking expected Precision@20",
            "hypothesis": "TF-IDF ranking exceeds the random Bethesda-match baseline",
            "changed_factor": "TF-IDF unigram and bigram cosine ranking",
            "acceptance_criterion": "pilot only; one query is not a final quality estimate",
            "mlflow_experiment_id": "2",
            "mlflow_experiment_name": "semantic-case-search",
            "mlflow_tracking_uri": "http://127.0.0.1:5000",
            "notebook_sha256": NOTEBOOK_SHA256,
            "scikit_learn_version": sklearn.__version__,
            "mlflow_version": mlflow.__version__,
        }
    )

    model_params = {
        f"tfidf__{name}": str(value)
        for name, value in vectorizer.get_params().items()
    }
    mlflow.log_params(model_params)

    mlflow.log_metrics(
        {
            "precision_at_20_same_bethesda_proxy": precision_at_20,
            "random_expected_precision_at_20": random_expected_precision_at_20,
            "lift_over_random_expected_precision_at_20": lift_over_random,
            "returned_cases": len(top_20_results),
        }
    )
    mlflow.set_tags(
        {
            "task": "similar_case_retrieval",
            "run_type": "single_query_pilot",
            "notebook": "notebooks/search/TF-IDF.ipynb",
            "dataset_version": DATASET_VERSION,
        }
    )

    if not is_allowed_artifact_source:
        raise ValueError(
            "Artifacts are enabled only for data/synthetic and "
            "data/processed/dataset_first.csv."
        )

    # Fit the same candidate index used by find_similar_cases.
    artifact_vectorizer = TfidfVectorizer(**vectorizer.get_params())
    artifact_matrix = artifact_vectorizer.fit_transform(
        candidate_cases_for_query["description"]
    )

    with tempfile.TemporaryDirectory() as temporary_dir:
        model_path = Path(temporary_dir) / "tfidf_index.joblib"
        joblib.dump(
            {
                "model_name": MODEL_NAME,
                "vectorizer": artifact_vectorizer,
                "tfidf_matrix": artifact_matrix,
            },
            model_path,
        )
        mlflow.log_artifact(str(model_path), artifact_path="model")

        examples_path = Path(temporary_dir) / "top20_examples.csv"
        top_20_results.to_csv(examples_path, index=False)
        mlflow.log_artifact(str(examples_path), artifact_path="retrieval_examples")
